In [ ]:
# Cell 1 — Load config and utilities
%run /home/jovyan/work/setup/config.py
import sys; sys.path.insert(0, "/home/jovyan/work")
from utils.dq import dq_check, write_dq_log
from utils.delta_utils import save_layer

In [ ]:
# Cell 2 — Read latest Bronze batch
# Bronze accumulates runs (append). Silver always rebuilds from the most recent batch.
from pyspark.sql.functions import col, trim, to_date, max as spark_max
from pyspark.sql.types import IntegerType, DoubleType

df_bronze_all = spark.read.format("delta").load(f"{BRONZE_PATH}/bronze_beverage_sales")
df_chan_all   = spark.read.format("delta").load(f"{BRONZE_PATH}/bronze_channel_group")

# Filter to latest ingest batch only
latest_ts   = df_bronze_all.agg(spark_max("_ingest_ts")).collect()[0][0]
latest_ts_c = df_chan_all.agg(spark_max("_ingest_ts")).collect()[0][0]

df_sales_b   = df_bronze_all.filter(col("_ingest_ts") == latest_ts)
df_channel_b = df_chan_all.filter(col("_ingest_ts") == latest_ts_c)

bronze_count = df_sales_b.count()
print(f"Latest Bronze batch: {bronze_count} rows (ts={latest_ts})")

In [ ]:
# Cell 3 — Transform sales: select+alias avoids Spark case-insensitive column collision
df_sales_clean = df_sales_b.select(
    to_date(col("DATE"), "M/d/yyyy").alias("full_date"),
    col("YEAR").cast(IntegerType()).alias("year"),
    col("MONTH").cast(IntegerType()).alias("month"),
    col("PERIOD").cast(IntegerType()).alias("period"),
    col("CE_BRAND_FLVR").cast(IntegerType()).alias("ce_brand_flvr"),
    col("dollar_volume_raw").cast(DoubleType()).alias("dollar_volume"),
    trim(col("BRAND_NM")).alias("brand_nm"),
    trim(col("TRADE_CHNL_DESC")).alias("trade_chnl_desc"),
    trim(col("Btlr_Org_LVL_C_Desc")).alias("region"),
    trim(col("PKG_CAT")).alias("pkg_cat"),
    trim(col("Pkg_Cat_Desc")).alias("pkg_cat_desc"),
    trim(col("TSR_PCKG_NM")).alias("tsr_pckg_nm"),
    trim(col("CHNL_GROUP")).alias("chnl_group"),
    col("_ingest_ts"),
    col("_source_file"),
)

In [ ]:
# Cell 4 — Transform channel
df_channel_clean = df_channel_b.select(
    trim(col("TRADE_CHNL_DESC")).alias("trade_chnl_desc"),
    trim(col("TRADE_GROUP_DESC")).alias("trade_group_desc"),
    trim(col("TRADE_TYPE_DESC")).alias("trade_type_desc"),
).dropDuplicates(["trade_chnl_desc"])

In [ ]:
# Cell 5 — Join
df_silver = df_sales_clean.join(df_channel_clean, on="trade_chnl_desc", how="left")

unmatched_count = df_sales_clean.join(df_channel_clean, on="trade_chnl_desc", how="left_anti").count()
silver_count    = df_silver.count()

print(f"Unmatched channels: {unmatched_count}")
print(f"Silver rows: {silver_count} | Bronze batch was: {bronze_count}")
assert silver_count == bronze_count, "Row count mismatch after join!"

In [ ]:
# Cell 6 — Overwrite Silver (full rebuild from latest Bronze snapshot)
save_layer(df_silver, "silver_beverage_sales_enriched", SILVER_PATH, PG_WRITE_PROPS,
           pg_schema="silver", delta_mode="overwrite", pg_mode="overwrite")
print("Silver layer complete")

In [ ]:
# Cell 7 — DQ Silver
import uuid
null_vol   = df_silver.filter(col("dollar_volume").isNull()).count()
null_brand = df_silver.filter(col("brand_nm").isNull()).count()

run_id = str(uuid.uuid4())
checks = [
    dq_check(run_id, "silver", "silver_beverage_sales_enriched",
             "row_count_parity",        str(bronze_count), str(silver_count)),
    dq_check(run_id, "silver", "silver_beverage_sales_enriched",
             "unmatched_channels_eq_0", "0", str(unmatched_count)),
    dq_check(run_id, "silver", "silver_beverage_sales_enriched",
             "null_dollar_volume",      "0", str(null_vol)),
    dq_check(run_id, "silver", "silver_beverage_sales_enriched",
             "null_brand_nm",           "0", str(null_brand)),
]
write_dq_log(spark, checks, GOLD_PATH)